# Breast Cancer Prediction
Predict breast cancer diagnosis from cell-nucleus features (Wisconsin dataset).

## Project Overview
Binary classification (M=malignant, B=benign) using the Wisconsin Breast Cancer dataset with 30 morphological features.

## Learning Objectives
- Work with a local medical CSV dataset
- Handle the diagnosis column (M/B encoding)
- Prioritize recall for cancer detection
- Compare gradient-boosting classifiers

## Problem Statement
Given 30 cell-nucleus measurements from a fine needle aspirate, predict whether the tumor is malignant (M) or benign (B).

## Why This Project Matters
Breast cancer is the most common cancer in women worldwide. ML-assisted screening can improve early detection rates and reduce unnecessary biopsies.

## Dataset Overview
| Property | Value |
|---|---|
| **Source** | Local: Wisconsin-bc-data.csv |
| **Target** | diagnosis (M/B) |
| **Rows** | 569 |
| **Features** | 30 cell-nucleus metrics (mean, SE, worst) |

## Dataset Source & License
Wisconsin Breast Cancer Diagnostic dataset from UCI ML Repository. Public domain.

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

## Imports

In [ ]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [ ]:
TARGET = 'diagnosis'

## Dataset Loading

In [ ]:
csv_path = os.path.join(SAVE_DIR, 'Wisconsin-bc-data.csv')
if not os.path.exists(csv_path):
    raise FileNotFoundError(f'Dataset not found: {csv_path}')
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
df.head()

## Data Validation

In [ ]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nTarget distribution:')
print(df[TARGET].value_counts())

# Drop id and unnamed columns
drop_cols = [c for c in df.columns if c.lower() in ('id', 'unnamed: 32') or 'unnamed' in c.lower()]
df = df.drop(columns=drop_cols, errors='ignore')
print(f'Shape after cleanup: {df.shape}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], color=['#2ecc71','#e74c3c'], edgecolor='black')
axes[0,0].set_title('Diagnosis Distribution')

for i, col in enumerate(['radius_mean', 'texture_mean', 'perimeter_mean']):
    if col in df.columns:
        ax = axes[(i+1)//2, (i+1)%2]
        for label in df[TARGET].unique():
            ax.hist(df[df[TARGET]==label][col], bins=25, alpha=0.5, label=str(label))
        ax.set_title(col)
        ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [ ]:
print(df[TARGET].value_counts())
print(f'\nMalignant rate: {(df[TARGET]=="M").mean():.2%}')

## Train/Test Split

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df[TARGET] = le.fit_transform(df[TARGET])
print(f'Encoded: {le.classes_} -> [0, 1]')

X = df.drop(columns=[TARGET])
# Fill any remaining NaN
X = X.fillna(X.median())
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## Preprocessing
Dropped ID column. Label-encoded M/B to 0/1. All features numeric.

## Feature Engineering
Using raw 30 cell-nucleus features. Tree models handle multicollinearity internally.

## Baseline Model

In [ ]:
bl = LogisticRegression(max_iter=5000, random_state=SEED)
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: Acc={accuracy_score(y_test, bl_pred):.4f}  F1={f1_score(y_test, bl_pred):.4f}')

## LazyPredict Benchmark

In [ ]:
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

## FLAML AutoML

In [ ]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Acc={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

## Additional Models: CatBoost, LightGBM, XGBoost

In [ ]:
models = {}
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

xgb_m = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb_m.fit(X_train, y_train)
models['XGBoost'] = xgb_m

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: Acc={accuracy_score(y_test, p):.4f}  F1={f1_score(y_test, p, average='weighted'):.4f}")

## Top 3 Model Selection

In [ ]:
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
results_df = pd.DataFrame(all_results).T.sort_values('F1', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

## Final Evaluation of Top 3

In [ ]:
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
    print(f"{name}: Acc={final_results[name]['Accuracy']:.4f}  F1={final_results[name]['F1']:.4f}")
    print(classification_report(y_test, p))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Accuracy', 'F1']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

## Error Analysis

In [ ]:
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    imp.tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('Feature Importance')
else:
    axes[1].text(0.5, 0.5, 'No feature importance', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

err = (preds != y_test.values).sum()
print(f'Errors: {err}, Error rate: {err/len(y_test):.4f}')

## Interpretation & Business Insight
Worst radius and concavity features most predictive. Larger, more irregular nuclei indicate malignancy â€” consistent with pathological knowledge.

## Limitations
- Small dataset (569 samples)
- Feature multicollinearity (mean/SE/worst correlated)
- No patient demographics or clinical staging

## How to Improve
- PCA or feature selection to handle multicollinearity
- Cross-validation with stratified folds
- Calibrated probability outputs for clinical use

## Production Considerations
- Recall optimization for cancer screening
- Model interpretability for clinical trust
- Regular revalidation with new data

## Common Mistakes
- Forgetting to drop the ID column
- Using accuracy on slightly imbalanced data
- Not prioritizing recall for cancer detection

## Mini Challenge
1. Apply PCA to reduce to 10 components, compare performance
2. Optimize threshold for 98%+ recall on malignant
3. Try StandardScaler and compare

## Final Summary
Predicted breast cancer diagnosis from cell-nucleus features. High accuracy achievable. Feature importance aligns with medical knowledge.

In [ ]:
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))